## Imports, Helpers, Parameters

### Imports

In [25]:
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
from textwrap import wrap
import numpy as np
import os
import re
import matplotlib.font_manager as fm




### Parameters

In [26]:
main_color = "#3a5f83"

font_dir = r"C:\Users\teddy\Downloads\OAIPR\Technical\AEI Data Other\Lato"

# Loop through every file in the folder
for font_file in os.listdir(font_dir):
    if font_file.lower().endswith(".ttf") and "lato" in font_file.lower():
        font_path = os.path.join(font_dir, font_file)
        fm.fontManager.addfont(font_path)

plt.rcParams["font.family"] = "Lato"
plt.rcParams["font.weight"] = "normal"

palette = sns.color_palette("colorblind")
# Put once at the TOP of your notebook/script (or just tweak this line)
sns.set_context("notebook", font_scale=1.0)  # was 1.2; smaller = less crowded

chart_size = (17, 9)

## Automation By Task Completion

### Load Data

In [ ]:
automation_tasks_imputed = pd.read_csv("../data/automation_tasks_imputed.csv")

### Top 15 % Occupation Automated

In [ ]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'pct_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='pct_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Most Automated Occupations by AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Occupation Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 % Occupation Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### % Major Occupational Category Automated

In [29]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "ai_task_comp_nat": "sum",
    "task_comp_nat": "sum"
}).reset_index()

# Compute automation percentages for each category
grouped["pct_automated_nat"] = (grouped["ai_task_comp_nat"] / grouped["task_comp_nat"]) * 100

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("pct_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='pct_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Major Occupational Categories Automated by AI Task Coverage", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Percent of Category Tasks Automated (%)", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0f}%"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "% Major Occupational Category Automated.png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation (National)

In [30]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_nat').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_nat', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage (National) ", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated (National)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Top 15 Workers Automated by Occupation (Utah)

In [31]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Get top 15 most automated occupations
top_15 = subset.nlargest(15, 'people_automated_ut').copy()

# Create combined label with title and major category
top_15['title_with_category'] = top_15.apply(
    lambda row: f"{row['title']} [{row['major_occ_category']}]",
    axis=1
)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use a professional color palette
palette = sns.color_palette("colorblind")
sns.barplot(data=top_15, x='people_automated_ut', y='title_with_category', 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=65)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.42)

# Professional titles and labels
ax.set_title("Top 15 Number of Workers Automated by Occupation by AI Task Coverage (Utah) ", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated (Utah)", fontsize=12, labelpad=10)
ax.set_ylabel("Occupation [Major Occupational Category]", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Top 15 Workers Automated by Occupation (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Workers Automated by Major Occupational Category (National)

In [32]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_nat": "sum",
}).reset_index()

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_nat", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_nat', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage (National)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category (National).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)

### Workers Automated by Major Occupational Category (Utah)

In [33]:
# Output directory for publication-ready charts
outdir = "../outputs/charts_for_sharing/report"
os.makedirs(outdir, exist_ok=True)

# Filter: no frequency increase (removes data quality issues)
no_freq_increase = automation_tasks_imputed['freq_sum_ai'] <= automation_tasks_imputed['freq_sum_eco']
subset = automation_tasks_imputed[no_freq_increase].copy()

# Aggregate to major occupation category level
grouped = subset.groupby("major_occ_category").agg({
    "people_automated_ut": "sum",
}).reset_index()

# Sort by automation percentage and get all categories (there aren't that many)
grouped = grouped.sort_values("people_automated_ut", ascending=False).reset_index(drop=True)

# Create figure
fig, ax = plt.subplots(figsize=chart_size)

# Use same professional color
sns.barplot(data=grouped, x='people_automated_ut', y="major_occ_category", 
            color=main_color, ax=ax)

# Wrap y labels
current_labels = [item.get_text() for item in ax.get_yticklabels()]
wrapped_labels = ['\n'.join(wrap(label, width=50)) for label in current_labels]
ax.set_yticks(ax.get_yticks())
ax.set_yticklabels(wrapped_labels, fontsize=10)
plt.subplots_adjust(left=0.38)

# Professional titles and labels
ax.set_title("Workers Automated by Major Occupational Category by AI Task Coverage (Utah)", 
             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel("Number of Workers Automated", fontsize=12, labelpad=10)
ax.set_ylabel("Major Occupational Category", fontsize=12, labelpad=10)

# Format x-axis as percentage
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Add grid for readability
ax.grid(axis='x', alpha=0.3, linestyle='--', linewidth=0.5)
ax.set_axisbelow(True)

# Add subtle note about methodology
fig.text(0.5, -0.025, 
         'Note: Assumes all tasks take the same amount of time. Analysis excludes 7 occupations with data quality issues. Based on 2025 O*NET task data, 2024 BLS employment & wage data, and AEI usage data.',
         ha='center', va='bottom', fontsize=8, style='italic', color='gray')

plt.tight_layout()

# Save with descriptive filename
filename = "Workers Automated by Major Occupational Category (Utah).png"
plt.savefig(os.path.join(outdir, filename), dpi=300, bbox_inches='tight')
plt.close(fig)